# Splitting by ensemble member

This is the step this cookbook changes from Unidata's notebooks, which split "randomly". Here the unit of the split is the **ensemble
member**, and the cost of getting it wrong is measured rather than assumed.


## Overview

1. Load the table with the test members of chapter 2 already named per forcing half.
2. Build member-grouped validation folds over the training members, and verify that no member sits on both sides of any fold.
3. Measure the dependence that would make a member-blind split leak: within members (lag-1) and between micro-siblings.
4. Score one model three ways — this cookbook's pipeline, transforms fit before the split, and a member-blind split — and read the gaps.

Shared code lives in `lensml.py` beside the chapters (parameters, transforms, ladder, scoring, paired test, register); `show()` prints a
function's source where a chapter introduces it.


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Chapter 2](02-target-and-forced-response.ipynb) | Helpful | the test draw and the two transforms |
| scikit-learn User Guide, §3.1 Cross-validation | Helpful | folds, groups |
| Autocorrelation | Helpful | lag-1, in one line |

- **Time to learn**: about 20 minutes; the contrast cell takes under a minute.
- **System requirements**: the `environment.yml` at the repository root; no account, no local data beyond the committed cache.


## Imports


In [ ]:
import itertools
import re

import numpy as np
from lensml import (
    ANOMALIES,
    BOOSTING_SETTINGS,
    FEATURES,
    N_FOLDS_MAX,
    N_REPEATS,
    RNG_SEED,
    WET_QUANTILE,
    WINDOW,
    member_folds,
    score_splits,
    setup,
    show,
)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import FixedThresholdClassifier, StratifiedKFold

In [ ]:
S = setup()  # the cache, the test members, the transforms (chapters 1-2)
train, test = S.train, S.test
X_tr, y_tr = train[FEATURES].values, train["wet"].values
g_tr = train["member_id"].values
print(
    f"train: {S.n_train_members} members / {len(train)} winters   "
    f"test: {test.member_id.nunique()} members / {len(test)} winters   "
    f"N_FOLDS = {S.n_folds}"
)

## Why the member is the unit

scikit-learn's guidance on grouped cross-validation: "all the samples in the validation fold come from groups that are not represented at
all in the paired training fold." Three reasons make the member the group:

1. **The question is about members never seen.** The score has to come from a realisation the model has not met; a member-blind split
   scores new *years* of members it already knows.
2. **The transforms.** The forced curve and the wet threshold are refit on each fold's training members (`score_splits` calls
   `fit_transforms` per fold). Fit before the split, held-out winters would help fit the curve and threshold that label them — the same
   leak as a `StandardScaler` fit on the whole table.
3. **Dependence is measured, not assumed.** Rodgers et al. 2021 §2.2 is why `WINDOW` starts late, but they give no figure for how much
   initial-condition memory is left after fifty years. The next cell prints the lag-1 autocorrelation inside each member, then the
   correlation between micro-siblings (members sharing a start year, as the CESM2 LE project page defines them — the `i1231`-style token
   in `member_id`) and between unrelated members, with a line that **fails** if siblings share more. Siblings are not grouped in the
   split; that line is what licenses it.

Two splits are made, both by member, both from the declared seed: the **test members** (chapter 2) and **member-grouped folds** over the
rest, `N_FOLDS = min(N_FOLDS_MAX, training members)`. The folds are stratified on the all-training labels, which decide only which members
share a fold; the labels scored inside a fold are refit on that fold's training members.


In [ ]:
show(member_folds)

In [ ]:
N_FOLDS = S.n_folds
print(
    f"base rate train {train.wet.mean():.3f}   (the test members' base rate is "
    "printed once, in chapter 6; nothing here reads their labels)"
)
print(
    f"N_FOLDS = min(N_FOLDS_MAX={N_FOLDS_MAX}, {S.n_train_members} training "
    f"members) = {N_FOLDS}"
)
for k, (a, b) in enumerate(
    member_folds(RNG_SEED, N_FOLDS).split(X_tr, y_tr, g_tr)
):
    assert not set(g_tr[a]) & set(g_tr[b]), f"fold {k} leaks members"
print(
    f"{N_FOLDS} member-grouped folds verified: no member appears in both the "
    "fitting and the scoring side of any fold"
)
if S.n_repeats_effective == 1:
    print(
        f"NOTE: leave-one-member-out; the trials use 1 fold draw, not {N_REPEATS}"
    )


def lag1(s):
    v = np.asarray(s, dtype=float)
    return float(np.corrcoef(v[:-1], v[1:])[0, 1])


dependence = (
    train.groupby("member_id")
    .agg(lag1_nino34_anom=("nino34_anom", lag1), lag1_wet=("wet", lag1))
    .round(3)
)
print(
    "\nlag-1 autocorrelation within training members (what consecutive winters "
    "of one member share):"
)
print(dependence.to_string())
print("mean:", dependence.mean().round(3).to_dict())

MICRO_STARTS = {"1231", "1251", "1281", "1301"}
start_year = {
    m: re.search(r"i(\d{4})p", m).group(1) for m in train.member_id.unique()
}
print(
    f"\ncorrelation between training members over {WINDOW[0]}–{WINDOW[1]} "
    "(micro-siblings share a start year; the memory check):"
)
for anom in ANOMALIES:
    wide = train.pivot(index="winter", columns="member_id", values=anom)
    corr = wide.corr()
    sib, unrel = [], []
    for a, b in itertools.combinations(wide.columns, 2):
        same = start_year[a] == start_year[b] and start_year[a] in MICRO_STARTS
        (sib if same else unrel).append(corr.loc[a, b])
    sib, unrel = np.array(sib), np.array(unrel)
    if len(sib) < 2 or len(unrel) < 2:
        print(f"{anom:12s} too few member pairs of one kind to compare")
        continue
    excess = sib.mean() - unrel.mean()
    se = np.sqrt(sib.var(ddof=1) / len(sib) + unrel.var(ddof=1) / len(unrel))
    if abs(excess) <= 2 * se:
        reading = (
            "PASS: no sibling excess detectable (|excess| ≤ 2 SE, SE a lower "
            "bound) — the member is taken as the unit"
        )
    else:
        reading = (
            "FAIL: siblings still move together — group the folds by start "
            "year, or start WINDOW later"
        )
    print(
        f"{anom:12s} micro-siblings {sib.mean():+.3f} ({len(sib)} pairs, SD "
        f"{sib.std(ddof=1):.3f}) · unrelated {unrel.mean():+.3f} ({len(unrel)} pairs, "
        f"SD {unrel.std(ddof=1):.3f}) · siblings − unrelated {excess:+.3f} (±{se:.3f}, "
        "a lower bound: the pairs share members, so they are not independent "
        f"draws) → {reading}"
    )

## The leak contrast: one model, three ways

One function scores every validation run in this cookbook. With `refit_transforms=True` it refits the forced curve and wet threshold on each
fold's training rows; with `False` it uses the columns fit once on all training members, before the folds.


In [ ]:
show(score_splits)

| Arm | Split unit | Forced curve and wet threshold fit on | A held-out score measures |
|---|---|---|---|
| A — this cookbook's pipeline | member | that fold's training members (refit in every fold) | new members |
| B — transforms fit before the split | member | all training members, before the folds | new members, scored with transforms they helped fit |
| C — member-blind split | winter (`StratifiedKFold`) | that fold's training winters | new years of members the model already knows |

B differs from A only in where the transforms are fit; C differs from A only in the split unit. The model is gradient boosting (chapter 4)
at the base-rate decision threshold, so that it *calls wet*. The cell's first line scores the same model at scikit-learn's default 0.5 and
prints how rarely it calls wet there: a precision resting on a handful of calls makes any gap a non-test, so below one wet call per hundred
winters the cell marks the gaps NON-TEST. Each arm prints its scores with per-fold SDs and its wet-call count, then its gaps against A
with the SD of the per-fold gaps — a rough scale; chapter 5's paired test is the measurement.

Note what this cell is, for chapter 5: a validation score for boosting at the base-rate threshold, on the same seeds chapter 5 uses. Once
it has been read, a trial of that model is no longer a blind one.


In [ ]:
CONTRAST_THRESHOLD = 1 - WET_QUANTILE  # the designed base rate
MIN_WET_CALL_SHARE = (
    0.01  # below this, precision and recall gaps are a NON-TEST
)


def contrast_model(seed):
    return FixedThresholdClassifier(
        GradientBoostingClassifier(**BOOSTING_SETTINGS, random_state=seed),
        threshold=CONTRAST_THRESHOLD,
        response_method="predict_proba",
    )


arms = {
    "A member split, transforms refit in each fold   ": dict(
        runs=[], cm=np.zeros((2, 2), dtype=int)
    ),
    "B member split, transforms fit before the split ": dict(
        runs=[], cm=np.zeros((2, 2), dtype=int)
    ),
    "C member-blind split, transforms refit in fold  ": dict(
        runs=[], cm=np.zeros((2, 2), dtype=int)
    ),
}
_s05, _cm05 = score_splits(
    GradientBoostingClassifier(**BOOSTING_SETTINGS, random_state=RNG_SEED),
    train,
    list(member_folds(RNG_SEED, N_FOLDS).split(X_tr, y_tr, g_tr)),
    refit_transforms=True,
)
print(
    f"for reference — the same model at sklearn's default threshold 0.5 (arm A's "
    f"split, one fold draw): wet calls {int(_cm05[:, 1].sum())} of {int(_cm05.sum())} "
    f"scored winters ({_cm05[:, 1].sum() / _cm05.sum():.2%}), recall "
    f"{_s05['recall'].mean():.3f} — too few calls to read a gap in"
)
for rep in range(S.n_repeats_effective):
    seed = RNG_SEED + rep
    by_member = list(member_folds(seed, N_FOLDS).split(X_tr, y_tr, g_tr))
    shuffled = StratifiedKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=seed
    )
    by_year = list(
        shuffled.split(X_tr, y_tr)
    )  # member-blind: winters shuffled
    plan = [(by_member, True), (by_member, False), (by_year, True)]
    for (label, store), (splits, refit) in zip(arms.items(), plan):
        s, cm = score_splits(
            contrast_model(seed), train, splits, refit_transforms=refit
        )
        store["runs"].append(s)
        store["cm"] += cm
METRICS = ["ap", "precision", "recall", "accuracy"]
pooled_arm = {
    label: {m: np.concatenate([s[m] for s in st["runs"]]) for m in METRICS}
    for label, st in arms.items()
}
ref = pooled_arm[next(iter(pooled_arm))]


def sd(v):
    return v.std(ddof=1) if len(v) > 1 else 0.0


print(
    f"boosting at threshold {CONTRAST_THRESHOLD:.3f} (the base rate; 0.5 is sklearn's "
    f"default), {S.n_repeats_effective} fold draw(s) × {N_FOLDS} folds per arm. Per arm: "
    "scores (mean ± per-fold SD) and wet calls; then gap = arm − A (mean ± SD of the "
    "per-fold gaps)"
)
for label, s in pooled_arm.items():
    cm = arms[label]["cm"]
    wet_calls, n_scored = int(cm[:, 1].sum()), int(cm.sum())
    share = wet_calls / n_scored
    print(
        f"{label} AP {s['ap'].mean():.3f}   precision {s['precision'].mean():.3f} ± "
        f"{sd(s['precision']):.3f}   recall {s['recall'].mean():.3f} ± "
        f"{sd(s['recall']):.3f}   accuracy {s['accuracy'].mean():.3f}   wet calls "
        f"{wet_calls} of {n_scored} scored winters ({share:.1%})"
    )
    gaps = {m: s[m] - ref[m] for m in ["ap", "precision", "recall"]}
    if share >= MIN_WET_CALL_SHARE:
        reading = (
            f"gaps readable (wet calls ≥ {MIN_WET_CALL_SHARE:.0%} of winters)"
        )
    else:
        reading = "precision/recall gaps a NON-TEST: read AP only"
    print(
        f"{'':{len(label)}} gap vs A: AP {gaps['ap'].mean():+.4f} ± {sd(gaps['ap']):.4f}"
        f"   precision {gaps['precision'].mean():+.4f} ± {sd(gaps['precision']):.4f}   "
        f"recall {gaps['recall'].mean():+.4f} ± {sd(gaps['recall']):.4f}   → {reading}"
    )

## Why the gaps are small here

Read each arm's `gap vs A` line against the SD printed beside it. On this table the leaks barely bite, for three reasons the printed lines
show:

- **No within-member persistence to memorise.** The `mean:` line under the lag-1 table gives `lag1_wet` near zero: one wet winter says
  almost nothing about the next in the same member, so a member-blind split (arm C) has little to learn from a member's other years. The
  input's lag-1 is negative, not positive — no run of similar autumns either.
- **The input carries no member identity.** With one detrended Niño 3.4 value per winter, nothing in a row tells the model which member it
  came from, so it cannot memorise a member even when it has seen one.
- **The transforms are smooth fits to many members.** A cubic in year through the mean of many training members, and one pooled quantile,
  barely move when one fold's members are left out — so fitting them before the split (arm B) leaks almost nothing.

A small gap here is a measurement on this design, not a licence: with a member-specific input (a member's own past winters, say) or
transforms fit on fewer members, the same cell is where a member split earns its keep.

```{admonition} Try it: the tidier pipeline
:class: tip
A colleague fits the forced curve and the wet threshold on the whole table, test members included, before any split. Say what the held-out
test in chapter 6 would then measure. Then read arm B's line: is its gap distinguishable from the per-fold SD beside it?
```


## Summary

The test members stay outside; validation folds are grouped by member and verified leak-free; the sibling check licenses taking the member
as the unit. Scoring one model three ways turned the two leaks into numbers. Here they are small, and the lag-1 table and the one-input
design say why.

### What's next?

[Chapter 4](04-baselines-and-the-ladder.ipynb) scores the simple rules and a three-rung ladder on these folds.

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| scikit-learn User Guide, §3.1 Cross-validation (grouped splits). https://scikit-learn.org/stable/ | member-grouped folds | BSD-3-Clause |
| scikit-learn User Guide, §12 Common pitfalls. https://scikit-learn.org/stable/common_pitfalls.html | transforms fit inside the split | BSD-3-Clause |
| Rodgers, K. B. et al. (2021). *Earth System Dynamics* 12, 1393–1411. https://doi.org/10.5194/esd-12-1393-2021 | §2.2: initial-condition memory | CC BY 4.0 |
| CESM2 Large Ensemble project page. https://www.cesm.ucar.edu/community-projects/lens2 | micro-members and start years | UCAR website terms of use |
| NSF Unidata, *Supervised Machine Learning Readiness*. https://github.com/Unidata/supervised-ml-readiness | the random split this chapter changes | Apache-2.0 |
